# 08 — Benchmark 2025: todos os campeões × ano intocado (só inferência)
Avaliação final do regime anual: checkpoints de 00–07 (treino 2024) previstos no ano de 2025, nunca tocado. Inclui o sazonal **lag-365** (copia a mesma data de 2024) e quebra mensal (sazonalidade do erro). Sem treino, sem tuning.

In [1]:
import json
import pickle
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from numpy.lib.stride_tricks import sliding_window_view

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
TR = ROOT / "dados" / "treino"
BM = ROOT / "dados" / "benchmark"
OUT = ROOT / "resultados" / "08-benchmark-2025"
(OUT / "figs").mkdir(parents=True, exist_ok=True)

L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
LN, HN = 2016, 288
FILES = {
    "ph": ("ef01-mogi-das-cruzes_ph_2024.csv", "ef01-mogi-das-cruzes_ph_2025.csv", "pH"),
    "od": ("ef01-mogi-das-cruzes_oxigenio-dissolvido_2024.csv", "ef01-mogi-das-cruzes_oxigenio-dissolvido_2025.csv", "Oxigênio Dissolvido (mg/L)"),
}
CKPTS = {
    "ph": {"lstnet": "02-lstnet-ph/modelos/lstnet_ph.pt", "patch": "04-patchtst-ph/modelos/patchtst_ph.pt",
           "dlin": "04-patchtst-ph/modelos/dlinear_ph.pt", "lgbm": "06-ensemble-ph/modelos/lgbm_steps.pkl",
           "dlres": "06-ensemble-ph/modelos/dlinear_res_ph.pt", "ens": "06-ensemble-ph/modelos/ensemble.json",
           "prophet": "00-baseline-ph/modelos/prophet_ph.json"},
    "od": {"lstnet": "03-lstnet-od/modelos/lstnet_od.pt", "patch": "05-patchtst-od/modelos/patchtst_od.pt",
           "dlin": "05-patchtst-od/modelos/dlinear_od.pt", "lgbm": "07-ensemble-od/modelos/lgbm_steps.pkl",
           "dlres": "07-ensemble-od/modelos/dlinear_res_od.pt", "ens": "07-ensemble-od/modelos/ensemble.json",
           "prophet": "01-baseline-od/modelos/prophet_od.json"},
}
for var, d in CKPTS.items():
    for k, rel in d.items():
        p = ROOT / "resultados" / rel
        if k == "prophet":
            print(f"{var}/{k}: {'OK' if p.exists() else 'AUSENTE (opcional, será pulado)'}")
        else:
            assert p.exists(), f"checkpoint ausente: {p} — rode o experimento correspondente antes!"
print("checkpoints obrigatórios OK | torch:", torch.__version__)

ph/prophet: OK
od/prophet: OK
checkpoints obrigatórios OK | torch: 2.14.0+cpu


## 1. Carga 2024 (fonte do lag-365) + 2025 (benchmark), limpeza idêntica

In [2]:
def ler(path, col):
    df = pd.read_csv(path, sep=";", decimal=",", encoding="windows-1252",
                     skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
    df = df.rename(columns={"Data hora": "ds", col: "y"}).sort_values("ds").reset_index(drop=True)
    idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
    s_raw = df.set_index("ds")["y"].reindex(idx)
    s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
    return s, s_raw

series = {}
for var, (f24, f25, col) in FILES.items():
    s24, _ = ler(TR / f24, col)
    s25, r25 = ler(BM / f25, col)
    series[var] = {"s24": s24, "s": s25, "raw": r25}
    print(f"{var}: treino {s24.shape} NaN={int(s24.isna().sum())} | benchmark {s25.shape} NaN={int(s25.isna().sum())}")

for var, d in series.items():
    s = d["s"]
    fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
    ax[0].plot(s.index, s.values, lw=0.3)
    ax[0].set_title(f"{var.upper()} 2025 — benchmark (ano intocado até aqui)")
    s.hist(bins=60, ax=ax[1])
    ax[1].set_title("Distribuição 2025")
    pd.Series(s.values, index=s.index).groupby(s.index.hour).mean().plot(ax=ax[2])
    ax[2].set_title("Ciclo diário médio 2025")
    ax[2].set_xlabel("hora")
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"01-eda-{var}.png")
print("figs salvas")

ph: treino (105121,) NaN=6408 | benchmark (104833,) NaN=544


od: treino (105121,) NaN=336 | benchmark (104833,) NaN=129


figs salvas


## 2. Janelamento do benchmark + âncoras diárias (ano todo)

In [3]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

bench = {}
for var, d in series.items():
    s = d["s"]
    v = s.to_numpy().astype(np.float32)
    W = sliding_window_view(v, L + H)
    ok = ~np.isnan(W).any(axis=1)
    W = W[ok]
    X, Y = W[:, :L], W[:, L:]
    ends = s.index[L + H - 1:][ok]
    di = np.where(ends.time == pd.Timestamp("23:55").time())[0]
    bench[var] = {"X": X, "Y": Y, "ends": ends, "di": di, "s": s}
    print(f"{var}: {len(X)} janelas | dias-âncora: {len(di)} ({ends[di[0]].date()} → {ends[di[-1]].date()})")

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

ph: 24589 janelas | dias-âncora: 86 (2025-01-31 → 2025-09-29)


od: 46556 janelas | dias-âncora: 163 (2025-01-31 → 2025-12-30)


## 3. Sazonal lag-365 (copia a mesma data de 2024; fallback p/ saz-288 onde 2024 falha)

In [4]:
lag365 = {}
for var, d in series.items():
    s24, X, Y, ends = d["s24"], bench[var]["X"], bench[var]["Y"], bench[var]["ends"]
    tgt_times = ends.values  # fim de cada alvo
    P = np.empty_like(Y)
    fb = 0
    s24v = s24.to_numpy()
    s24i = s24.index
    for k in range(len(Y)):
        e = ends[k]
        try:
            src_end = s24i.get_loc(pd.Timestamp(year=2024, month=e.month, day=e.day,
                                                hour=e.hour, minute=e.minute))
            src = s24v[src_end-287:src_end+1]
        except KeyError:
            src = None
        if src is None or np.isnan(src).any():
            src = X[k][L-SEASON:L]  # fallback honesto: saz-288
            fb += 1
        P[k] = src
    lag365[var] = P
    print(f"{var}: lag-365 pronto | fallback saz-288 em {fb}/{len(Y)} janelas ({100*fb/len(Y):.1f}%)")
    print(f"  lag-365 rolante:", {k: round(v, 4) for k, v in metricas(Y, P).items()})

ph: lag-365 pronto | fallback saz-288 em 300/24589 janelas (1.2%)
  lag-365 rolante: {'MAE': 0.464, 'RMSE': 0.5463, 'MAPE': 7.5154, 'sMAPE': 7.9273}


od: lag-365 pronto | fallback saz-288 em 600/46556 janelas (1.3%)


  lag-365 rolante: {'MAE': 0.9252, 'RMSE': 1.1881, 'MAPE': 16.8312, 'sMAPE': 19.6396}


## 4. Arquiteturas (cópias fiéis de 02/03/04) + reload dos checkpoints

In [5]:
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

class PatchTST(nn.Module):
    def __init__(self):
        super().__init__()
        self.N = (LN - 48) // 24 + 1
        self.proj = nn.Linear(48, 64)
        self.pos = nn.Parameter(torch.randn(1, self.N, 64) * 0.02)
        layer = nn.TransformerEncoderLayer(64, 4, 128, 0.1, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 3)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(self.N * 64, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        z = self.proj(xn.unfold(1, 48, 24)) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu

class DLinearLite(nn.Module):
    def __init__(self, k=25, residual=False):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, HN)
        self.lin_s = nn.Linear(LN, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
        self.residual = residual
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        yn = (y - self.beta) / self.gamma.clamp_min(1e-3) * sg
        return yn if self.residual else yn + mu

def load_state(cls, rel, **kw):
    m = cls(**kw).to("cpu")
    m.load_state_dict(torch.load(ROOT / "resultados" / rel, map_location="cpu", weights_only=False)["state"])
    return m.eval()

## 5. Inferência por variável (cheap + lag365 + 6 modelos; Prophet se houver artefato)

In [6]:
def base_feats(Xb, E):
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em

def hour_sincos(em, j):
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

resultados = {}
for var in ["ph", "od"]:
    t0 = time.time()
    X, Y, ends = bench[var]["X"], bench[var]["Y"], bench[var]["ends"]
    preds = cheap_preds(X)
    preds["sazonal_lag365"] = lag365[var]
    s = bench[var]["s"]
    val5 = s.to_numpy().astype(np.float32)
    SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
    COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
    Wln = sliding_window_view(val5, LN)
    Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), LN, axis=0).transpose(0, 2, 1).astype(np.float32)
    rowln = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H)) - LN + 1
    C = CKPTS[var]
    lstnet = load_state(LSTNet1D, C["lstnet"])
    patch = load_state(PatchTST, C["patch"])
    dlin = load_state(DLinearLite, C["dlin"])
    dlres = load_state(DLinearLite, C["dlres"], residual=True)
    with open(ROOT / "resultados" / C["lgbm"], "rb") as f:
        lgbms = pickle.load(f)
    ens = json.load(open(ROOT / "resultados" / C["ens"]))["pesos"]
    w = [ens["sazonal"], ens["lstnet"], ens["lgbm"], ens["dlres"]]

    @torch.no_grad()
    def fwd1(model, idxs, tod=None, batch=512):
        outs = []
        for b in range(0, len(idxs), batch):
            xb = torch.from_numpy(Wln[rowln[idxs[b:b+batch]]])
            if tod is None:
                outs.append(model(xb).numpy())
            else:
                outs.append(model(xb, torch.from_numpy(tod[rowln[idxs[b:b+batch]]])).numpy())
        return np.concatenate(outs)

    idx = np.arange(len(X))
    Pn = fwd1(lstnet, idx, Tln)
    Pt = fwd1(patch, idx)
    Dl = fwd1(dlin, idx)
    F, em = base_feats(X, ends)
    S = preds["sazonal_naive_288"]
    Gb = np.empty((len(X), H), dtype=np.float32)
    for j, m in enumerate(lgbms):
        sh, ch = hour_sincos(em, j)
        Gb[:, j] = S[:, j] + m.predict(np.column_stack([F, sh, ch]))
    Dr = S + fwd1(dlres, idx)
    En = w[0]*S + w[1]*Pn + w[2]*Gb + w[3]*Dr
    preds.update({"lstnet": Pn, "patchtst": Pt, "dlinear": Dl, "lgbm": Gb, "dlres": Dr, "ens": En})
    Pp = (ROOT / "resultados" / C["prophet"])
    if Pp.exists():
        from prophet.serialize import model_from_json
        m = model_from_json(Pp.read_text())
        fmap = m.predict(pd.DataFrame({"ds": s.index})).set_index("ds")["yhat"]
        E = ends
        preds["prophet"] = np.stack([[fmap.loc[d - pd.Timedelta(minutes=5*(H-1-h))] for h in range(H)] for d in E])
        print(f"{var}: prophet incluído")
    else:
        print(f"{var}: prophet ausente — pulado")
    resultados[var] = preds
    print(f"{var}: inferência em {time.time()-t0:.0f}s | modelos: {sorted(preds)}")
    del F, Gb, X, Y

Importing plotly failed. Interactive plots will not work.


ph: prophet incluído
ph: inferência em 181s | modelos: ['dlinear', 'dlres', 'ens', 'lgbm', 'lstnet', 'media_movel_288', 'patchtst', 'persistencia', 'prophet', 'sazonal_lag365', 'sazonal_naive_288']


od: prophet incluído
od: inferência em 331s | modelos: ['dlinear', 'dlres', 'ens', 'lgbm', 'lstnet', 'media_movel_288', 'patchtst', 'persistencia', 'prophet', 'sazonal_lag365', 'sazonal_naive_288']


## 6. Tabelas (rolante + dias-âncora + por dia + por mês)

In [7]:
for var in ["ph", "od"]:
    X, Y, ends, di = bench[var]["X"], bench[var]["Y"], bench[var]["ends"], bench[var]["di"]
    tab = pd.DataFrame({m: metricas(Y, p) for m, p in resultados[var].items()}).T.round(4)
    tab.to_csv(OUT / f"metricas_benchmark_{var}.csv")
    print(f"=== {var} benchmark rolante ({len(X)} origens) ===")
    print(tab.to_string())
    Yd = Y[di]
    tab_d = pd.DataFrame({m: metricas(Yd, p[di]) for m, p in resultados[var].items()}).T.round(4)
    tab_d.to_csv(OUT / f"metricas_diaria_{var}.csv")
    print(f"=== {var} dias-âncora ({len(di)}) ===")
    print(tab_d.to_string())
    por_dia = pd.DataFrame({m: [mae(Yd[k:k+1], p[di][k:k+1]) for k in range(len(di))]
                            for m, p in resultados[var].items()},
                           index=[str(ends[i].date()) for i in di])
    por_dia.to_csv(OUT / f"metricas_por_dia_{var}.csv")
    meses = pd.to_datetime(por_dia.index).month
    por_mes = por_dia.groupby(meses).mean().round(4)
    nomes_mes = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
    por_mes.index = [nomes_mes[m-1] for m in por_mes.index]  # só meses com âncora (outages removem meses)
    por_mes.to_csv(OUT / f"metricas_por_mes_{var}.csv")
    print(f"=== {var} MAE médio por mês ===")
    print(por_mes.to_string())
    print(f"\nRégua benchmark {var}: {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}\n")

=== ph benchmark rolante (24589 origens) ===
                      MAE    RMSE     MAPE    sMAPE
persistencia       0.0844  0.1133   1.3690   1.3686
sazonal_naive_288  0.0597  0.0823   0.9735   0.9729
media_movel_288    0.0715  0.0926   1.1605   1.1598
sazonal_lag365     0.4640  0.5463   7.5154   7.9273
lstnet             0.0529  0.0732   0.8629   0.8620
patchtst           0.0688  0.0949   1.1200   1.1196
dlinear            0.0600  0.0856   0.9772   0.9764
lgbm               0.0671  0.0907   1.0895   1.0931
dlres              0.0710  0.1073   1.1539   1.1537
ens                0.0509  0.0704   0.8293   0.8290
prophet            1.8997  2.0996  30.7587  38.0236
=== ph dias-âncora (86) ===
                      MAE    RMSE     MAPE    sMAPE
persistencia       0.0902  0.1218   1.4695   1.4550
sazonal_naive_288  0.0596  0.0821   0.9718   0.9713
media_movel_288    0.0702  0.0909   1.1392   1.1388
sazonal_lag365     0.4620  0.5449   7.4820   7.8916
lstnet             0.0529  0.0732   0.8624 

=== ph MAE médio por mês ===
     persistencia  sazonal_naive_288  media_movel_288  sazonal_lag365  lstnet  patchtst  dlinear    lgbm   dlres     ens  prophet
jan        0.0592             0.0631           0.0517          0.0781  0.0613    0.1755   0.0880  0.0712  0.0766  0.0568   0.3536
fev        0.0649             0.0916           0.0757          0.2350  0.0724    0.0845   0.0703  0.0873  0.0767  0.0746   0.3895
mar        0.0729             0.0473           0.0603          0.1365  0.0371    0.0496   0.0384  0.0517  0.0398  0.0374   1.0152
abr        0.0440             0.0337           0.0452          0.0980  0.0177    0.0265   0.0228  0.0311  0.0332  0.0184   1.0575
mai        0.0471             0.0414           0.0394          0.5840  0.0380    0.0447   0.0438  0.0411  0.0454  0.0356   1.4554
ago        0.1039             0.0550           0.0731          0.6480  0.0486    0.0531   0.0465  0.0813  0.0490  0.0453   2.6842
set        0.1311             0.0785           0.0961        

=== od benchmark rolante (46556 origens) ===
                      MAE    RMSE     MAPE     sMAPE
persistencia       0.5258  0.7067   9.2976    9.1593
sazonal_naive_288  0.2714  0.4015   4.9908    4.9220
media_movel_288    0.4545  0.5645   8.1220    7.9713
sazonal_lag365     0.9252  1.1881  16.8312   19.6396
lstnet             0.2127  0.3220   3.9994    3.9164
patchtst           0.2213  0.3226   4.1261    4.0517
dlinear            0.2325  0.3416   4.3016    4.2665
lgbm               0.2943  0.4197   5.3455    5.3049
dlres              0.2286  0.3407   4.2368    4.2009
ens                0.2107  0.3220   3.9523    3.8639
prophet            5.0446  5.4095  83.5542  143.4268
=== od dias-âncora (163) ===
                      MAE    RMSE     MAPE     sMAPE
persistencia       0.6230  0.7991  11.5018   10.5271
sazonal_naive_288  0.2732  0.4043   5.0411    4.9660
media_movel_288    0.4459  0.5556   7.9869    7.8414
sazonal_lag365     0.9264  1.1900  16.8870   19.7181
lstnet             0.2099

=== od MAE médio por mês ===
     persistencia  sazonal_naive_288  media_movel_288  sazonal_lag365  lstnet  patchtst  dlinear    lgbm   dlres     ens  prophet
jan        0.3523             0.2780           0.2091          0.2414  0.5256    0.5527   0.2997  0.1913  0.3914  0.4712   1.2669
fev        0.1578             0.2170           0.2008          0.3347  0.2234    0.2034   0.2252  0.2378  0.2034  0.2058   1.4009
mar        0.9685             0.3040           0.7023          2.4049  0.1967    0.2388   0.2633  0.4023  0.2538  0.1864   0.8975
mai        0.3781             0.3714           0.3663          1.4110  0.3060    0.3241   0.3603  0.3318  0.3523  0.3078   2.1151
jun        0.3830             0.1786           0.2162          1.2603  0.1487    0.1633   0.1213  0.1740  0.1138  0.1631   4.4810
jul        0.4074             0.1410           0.2629          1.0809  0.1118    0.1179   0.1108  0.1489  0.1117  0.1091   5.0393
ago        0.9665             0.1793           0.6305        

## 7. Figuras

In [8]:
for var in ["ph", "od"]:
    tab = pd.read_csv(OUT / f"metricas_benchmark_{var}.csv", index_col=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    tab["MAE"].sort_values().plot.barh(ax=ax)
    ax.set_title(f"MAE benchmark 2025 — {var} (menor = melhor)")
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"04-mae-{var}.png")

    pord = pd.read_csv(OUT / f"metricas_por_dia_{var}.csv", index_col=0)
    fig, ax = plt.subplots(figsize=(12, 3.5))
    for col in ["sazonal_lag365", "sazonal_naive_288", "lstnet", "ens"]:
        if col in pord.columns:
            ax.plot(pd.to_datetime(pord.index), pord[col], lw=1.1, label=col)
    ax.set_title(f"{var} — MAE por dia em 2025")
    ax.legend(fontsize=8); fig.autofmt_xdate()
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"05-diaria-{var}.png")

    porm = pd.read_csv(OUT / f"metricas_por_mes_{var}.csv", index_col=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    for col in ["sazonal_lag365", "sazonal_naive_288", "lstnet", "ens"]:
        if col in porm.columns:
            ax.plot(porm.index, porm[col], marker="o", ms=3, lw=1.2, label=col)
    ax.set_title(f"{var} — MAE médio por mês em 2025 (sazonalidade do erro)")
    ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"06-mensal-{var}.png")

    X, Y, ends, di = bench[var]["X"], bench[var]["Y"], bench[var]["ends"], bench[var]["di"]
    ks = [0, len(di)//2, -1]
    fig, axes = plt.subplots(3, 1, figsize=(12, 9))
    for ax, k in zip(axes, ks):
        tf = pd.date_range(ends[di[k]] - pd.Timedelta(minutes=5*(H-1)), ends[di[k]], freq="5min")
        ax.plot(tf, Y[di[k]], "k-", lw=1.2, label="real")
        ax.plot(tf, resultados[var]["sazonal_lag365"][di[k]], "--", lw=1, label="lag-365")
        ax.plot(tf, resultados[var]["sazonal_naive_288"][di[k]], ":", lw=1, label="saz-288")
        ax.plot(tf, resultados[var]["ens"][di[k]], lw=1, alpha=0.9, label="ens")
        ax.set_title(f"{var} dia {ends[di[k]].date()}")
        ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"07-exemplos-{var}.png")
print("figs salvas")

figs salvas


## 8. Veredito
Réguas do benchmark 2025 acima (célula 6). Este notebook não treina nada.